# Building a USA Artifact from GBD Data & Comparing Approaches

This notebook documents two approaches to building a USA-level artifact:

1. **Population-weighted aggregation** (completed): Combine 51 state-level artifacts,
   weighting each demographic cell by its state population.
2. **Direct GBD query** (documented here): Use the `make_artifacts` pipeline to pull
   USA-level data directly from the GBD database.

The GBD approach requires IHME cluster access and cannot run locally.
This notebook documents the exact steps, then provides comparison code
that works once both artifacts exist.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ARTIFACT_DIR = Path('../src/vivarium_nih_us_cvd/artifacts')
POPWT_PATH = ARTIFACT_DIR / 'united_states_of_america.hdf'  # population-weighted
GBD_PATH = ARTIFACT_DIR / 'united_states_of_america_gbd.hdf'  # direct GBD (when available)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## Part 1: How to Build the USA Artifact via the GBD Pipeline

The `make_artifacts` CLI and its underlying code in
`src/vivarium_nih_us_cvd/tools/make_artifacts.py` pull data from the
GBD database using `vivarium_inputs`, `vivarium_gbd_access`, and
`gbd_mapping`. These packages are only available on the IHME cluster.

### Prerequisites

1. Access to the IHME computing cluster
2. A conda environment with the data dependencies installed:
   ```bash
   conda create -n artifact_env python=3.11
   conda activate artifact_env
   cd /path/to/vivarium_nih_us_cvd
   pip install -e .[data]
   ```

### Step 1: Add USA to the location list

Edit `src/vivarium_nih_us_cvd/constants/metadata.py` and add
`"United States of America"` to the `LOCATIONS` list:

```python
LOCATIONS = [
    "Alabama",
    "Alaska",
    # ... existing states ...
    "Wyoming",
    "United States of America",  # <-- add this
]
```

### Step 2: Add USA entries to local CSV files

Two local data files contain state-level data only and need USA entries:

1. **`src/vivarium_nih_us_cvd/data/hf_props.csv`** — Heart failure
   proportions by state, sex, and age.
2. **`src/vivarium_nih_us_cvd/data/state_medication_real_data_v3.csv`** —
   Medication coverage/adherence data by state.

For `hf_props.csv`, you can compute population-weighted USA rows from
the state data. For `state_medication_real_data_v3.csv`, similarly
aggregate or use national-level source data.

### Step 3: Build the artifact (without PAFs)

On the IHME cluster:

```bash
conda activate artifact_env
cd /path/to/vivarium_nih_us_cvd

# Build for USA only, ignoring PAFs initially
make_artifacts -vvv --pdb \
    -l "United States of America" \
    --ignore-pafs \
    -o src/vivarium_nih_us_cvd/artifacts/
```

This calls `build_single_location_artifact()` which iterates over
all 14 key groups in `data_keys.MAKE_ARTIFACT_KEY_GROUPS`:
- POPULATION, ISCHEMIC_STROKE, IHD_AND_HF
- LDL_C, SBP, BMI, FPG
- LDLC_MEDICATION_ADHERENCE, SBP_MEDICATION_ADHERENCE
- OUTREACH, POLYPILL, MEDIATION
- JOINT_PAFS, MEDICATION_COVERAGE

Each key calls `loader.get_data()` which dispatches to the appropriate
loader function (most use `vivarium_inputs.interface.get_measure()`
to query GBD).

### Step 4: Calculate PAFs

PAFs are computed via a dedicated simulation using
`paf_calculation.yaml`. This runs 100,000 simulants for 1 day
using `PAFCalculationRiskEffect` and `JointPAFObserver` components.

```bash
conda activate sim_env  # environment with .[dev] installed

# Create output directory
mkdir -p src/vivarium_nih_us_cvd/artifacts/paf_calculation

# Run PAF simulation for USA
simulate run_paf \
    src/vivarium_nih_us_cvd/model_specifications/paf_calculation.yaml \
    --artifact-path src/vivarium_nih_us_cvd/artifacts/united_states_of_america_gbd.hdf
```

Alternatively, using `psimulate` on the cluster for parallelism:
```bash
psimulate run \
    src/vivarium_nih_us_cvd/model_specifications/paf_calculation.yaml \
    src/vivarium_nih_us_cvd/model_specifications/branches/paf_scenarios.yaml \
    -o src/vivarium_nih_us_cvd/artifacts/ \
    --max-workers 5000 -vvv --pdb -m 20 -r 1:00:00 -P proj_simscience_prod
```

### Step 5: Append PAFs to the artifact

```bash
conda activate artifact_env
make_artifacts --pdb -vvv \
    -a -l "United States of America" \
    -o src/vivarium_nih_us_cvd/artifacts/
```

The `load_joint_pafs()` function in `loader.py` reads the PAF
simulation output from `artifacts/paf_calculation/<latest>/output.hdf`
and formats it for the artifact.

### Step 6: Rename the artifact

To keep both artifacts separate for comparison:

```bash
mv src/vivarium_nih_us_cvd/artifacts/united_states_of_america.hdf \
   src/vivarium_nih_us_cvd/artifacts/united_states_of_america_gbd.hdf
```

Then restore the population-weighted artifact to the original name.

---
## Part 2: Visual Comparison of the Two Artifacts

The code below compares the population-weighted artifact with the
GBD-direct artifact. **It will only run once both files exist.**

In [2]:
# Check which artifacts are available
popwt_exists = POPWT_PATH.exists()
gbd_exists = GBD_PATH.exists()

print(f'Population-weighted artifact: {"FOUND" if popwt_exists else "MISSING"} ({POPWT_PATH})')
print(f'GBD-direct artifact:          {"FOUND" if gbd_exists else "MISSING"} ({GBD_PATH})')

if not gbd_exists:
    print('\n*** GBD artifact not found. Follow the steps in Part 1 to build it. ***')
    print('*** The comparison cells below will be skipped. ***')

Population-weighted artifact: FOUND (../src/vivarium_nih_us_cvd/artifacts/united_states_of_america.hdf)
GBD-direct artifact:          MISSING (../src/vivarium_nih_us_cvd/artifacts/united_states_of_america_gbd.hdf)

*** GBD artifact not found. Follow the steps in Part 1 to build it. ***
*** The comparison cells below will be skipped. ***


### 2.1 Key Inventory Comparison

In [ ]:
if gbd_exists:
    with pd.HDFStore(str(POPWT_PATH), 'r') as s1, pd.HDFStore(str(GBD_PATH), 'r') as s2:
        keys_pw = set(s1.keys())
        keys_gbd = set(s2.keys())

    print(f'Pop-weighted keys: {len(keys_pw)}')
    print(f'GBD-direct keys:   {len(keys_gbd)}')
    print(f'\nIn pop-weighted only: {keys_pw - keys_gbd or "none"}')
    print(f'In GBD-direct only:   {keys_gbd - keys_pw or "none"}')
else:
    print('Skipped (GBD artifact not available)')

### 2.2 Population Structure Comparison

In [ ]:
if gbd_exists:
    pw_pop = pd.read_hdf(POPWT_PATH, '/population/structure').reset_index()
    gbd_pop = pd.read_hdf(GBD_PATH, '/population/structure').reset_index()
    if 'location' in pw_pop.columns:
        pw_pop = pw_pop.drop(columns=['location'])
    if 'location' in gbd_pop.columns:
        gbd_pop = gbd_pop.drop(columns=['location'])

    print(f'Pop-weighted total: {pw_pop["value"].sum():,.0f}')
    print(f'GBD-direct total:   {gbd_pop["value"].sum():,.0f}')
    print(f'Difference:         {pw_pop["value"].sum() - gbd_pop["value"].sum():,.0f}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for sex, ax in zip(['Female', 'Male'], axes):
        pw = pw_pop[pw_pop['sex'] == sex].sort_values('age_start')
        gb = gbd_pop[gbd_pop['sex'] == sex].sort_values('age_start')
        ages = pw['age_start'].values
        width = 0.35
        x = np.arange(len(ages))
        ax.barh(x - width/2, pw['value']/1e6, width, label='Pop-weighted', color='steelblue')
        ax.barh(x + width/2, gb['value']/1e6, width, label='GBD-direct', color='coral')
        labels = [f"{int(a)}-{int(b)}" for a, b in zip(pw['age_start'], pw['age_end'])]
        ax.set_yticks(x)
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_xlabel('Population (millions)')
        ax.set_title(sex)
        ax.legend(fontsize=8)
    fig.suptitle('Population Structure: Pop-Weighted vs GBD-Direct', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Skipped (GBD artifact not available)')

### 2.3 Disease Rate Comparisons

In [ ]:
def load_rate(path, key, draw='draw_0'):
    """Load a rate from artifact and return as age-sex DataFrame."""
    df = pd.read_hdf(path, key)
    if draw in df.columns:
        df = df[[draw]].rename(columns={draw: 'value'})
    return df.reset_index()


def compare_rates(key, title, ax):
    """Plot pop-weighted vs GBD-direct for a given key."""
    for label, path, color, ls in [
        ('Pop-weighted', POPWT_PATH, 'steelblue', '-'),
        ('GBD-direct', GBD_PATH, 'coral', '--'),
    ]:
        df = load_rate(path, key)
        for sex, marker in [('Female', 'o'), ('Male', 's')]:
            sub = df[df['sex'] == sex].sort_values('age_start')
            ax.plot(sub['age_start'], sub['value'],
                    label=f'{label} ({sex[0]})', color=color,
                    linestyle='-' if sex == 'Female' else '--',
                    linewidth=2 if label == 'Pop-weighted' else 1.5,
                    alpha=0.9 if label == 'Pop-weighted' else 0.7)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Age')
    ax.set_ylabel('Rate')
    ax.legend(fontsize=7)
    ax.set_xlim(25, 100)

In [ ]:
if gbd_exists:
    rate_keys = {
        'AMI Incidence': '/cause/acute_myocardial_infarction/incidence_rate',
        'Ischemic Stroke Incidence': '/cause/ischemic_stroke/incidence_rate',
        'All-cause Mortality': '/cause/all_causes/cause_specific_mortality_rate',
        'HF Residual Incidence': '/cause/heart_failure_residual/incidence_rate',
        'Post MI Prevalence': '/cause/post_myocardial_infarction/prevalence',
        'HF IHD Prevalence': '/cause/heart_failure_from_ischemic_heart_disease/prevalence',
    }

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    for ax, (title, key) in zip(axes.flat, rate_keys.items()):
        compare_rates(key, title, ax)
    fig.suptitle('Disease Rates: Pop-Weighted vs GBD-Direct (draw 0)', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Skipped (GBD artifact not available)')

### 2.4 Risk Factor Exposure Comparisons

In [ ]:
if gbd_exists:
    exposure_keys = {
        'SBP (mmHg)': '/risk_factor/high_systolic_blood_pressure/exposure',
        'LDL-C': '/risk_factor/high_ldl_cholesterol/exposure',
        'BMI': '/risk_factor/high_body_mass_index_in_adults/exposure',
        'FPG': '/risk_factor/high_fasting_plasma_glucose/exposure',
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (title, key) in zip(axes.flat, exposure_keys.items()):
        compare_rates(key, title, ax)
    fig.suptitle('Risk Factor Exposures: Pop-Weighted vs GBD-Direct (draw 0)', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Skipped (GBD artifact not available)')

### 2.5 Joint PAF Comparisons

In [ ]:
if gbd_exists:
    paf_key = '/risk_factor/joint_mediated_risks/population_attributable_fraction'
    pw_pafs = pd.read_hdf(POPWT_PATH, paf_key)
    pw_pafs = pw_pafs[['draw_0']].rename(columns={'draw_0': 'value'}).reset_index()
    gbd_pafs = pd.read_hdf(GBD_PATH, paf_key)
    gbd_pafs = gbd_pafs[['draw_0']].rename(columns={'draw_0': 'value'}).reset_index()

    entities = pw_pafs['affected_entity'].unique()
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    for ax, entity in zip(axes.flat, entities):
        for label, df, color, ls in [
            ('Pop-weighted', pw_pafs, 'steelblue', '-'),
            ('GBD-direct', gbd_pafs, 'coral', '--'),
        ]:
            sub = df[(df['affected_entity'] == entity) & (df['sex'] == 'Female')]
            sub = sub.sort_values('age_start')
            ax.plot(sub['age_start'], sub['value'], label=label,
                    color=color, linewidth=2, linestyle=ls)
        ax.set_title(entity.replace('_', ' '), fontsize=9)
        ax.set_xlabel('Age')
        ax.set_ylabel('PAF')
        ax.legend(fontsize=7)
        ax.set_xlim(25, 100)

    fig.suptitle('Joint PAFs: Pop-Weighted vs GBD-Direct (draw 0, Female)', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Skipped (GBD artifact not available)')

### 2.6 Quantitative Summary: Relative Differences

For each key and demographic cell, compute the relative difference
between the two approaches.

In [ ]:
if gbd_exists:
    summary_keys = {
        'AMI Incidence': '/cause/acute_myocardial_infarction/incidence_rate',
        'Stroke Incidence': '/cause/ischemic_stroke/incidence_rate',
        'ACMR': '/cause/all_causes/cause_specific_mortality_rate',
        'HF Residual Inc': '/cause/heart_failure_residual/incidence_rate',
        'SBP Exposure': '/risk_factor/high_systolic_blood_pressure/exposure',
        'LDL-C Exposure': '/risk_factor/high_ldl_cholesterol/exposure',
        'BMI Exposure': '/risk_factor/high_body_mass_index_in_adults/exposure',
        'FPG Exposure': '/risk_factor/high_fasting_plasma_glucose/exposure',
    }

    results = []
    for name, key in summary_keys.items():
        pw = load_rate(POPWT_PATH, key)
        gb = load_rate(GBD_PATH, key)
        # Merge on common index columns
        merge_cols = [c for c in ['sex', 'age_start', 'age_end', 'year_start', 'year_end']
                      if c in pw.columns and c in gb.columns]
        merged = pw.merge(gb, on=merge_cols, suffixes=('_pw', '_gbd'))
        merged['rel_diff'] = (merged['value_pw'] - merged['value_gbd']) / merged['value_gbd'].replace(0, np.nan)
        results.append({
            'Measure': name,
            'Mean Rel Diff (%)': merged['rel_diff'].mean() * 100,
            'Median Rel Diff (%)': merged['rel_diff'].median() * 100,
            'Max Abs Rel Diff (%)': merged['rel_diff'].abs().max() * 100,
            'Cells Compared': len(merged),
        })

    summary_df = pd.DataFrame(results)
    print(summary_df.to_string(index=False, float_format='%.3f'))
else:
    print('Skipped (GBD artifact not available)')

### 2.7 Scatter Plots: Pop-Weighted vs GBD-Direct

In [ ]:
if gbd_exists:
    scatter_keys = {
        'AMI Incidence': '/cause/acute_myocardial_infarction/incidence_rate',
        'Stroke Incidence': '/cause/ischemic_stroke/incidence_rate',
        'SBP Exposure': '/risk_factor/high_systolic_blood_pressure/exposure',
        'LDL-C Exposure': '/risk_factor/high_ldl_cholesterol/exposure',
    }

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    for ax, (title, key) in zip(axes.flat, scatter_keys.items()):
        pw = load_rate(POPWT_PATH, key)
        gb = load_rate(GBD_PATH, key)
        merge_cols = [c for c in ['sex', 'age_start', 'age_end', 'year_start', 'year_end']
                      if c in pw.columns and c in gb.columns]
        merged = pw.merge(gb, on=merge_cols, suffixes=('_pw', '_gbd'))

        for sex, color in [('Female', 'coral'), ('Male', 'steelblue')]:
            sub = merged[merged['sex'] == sex]
            ax.scatter(sub['value_gbd'], sub['value_pw'], alpha=0.6,
                       color=color, label=sex, s=20)

        # Perfect agreement line
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
                max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, 'k--', alpha=0.5, linewidth=1, label='y=x')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel('GBD-direct')
        ax.set_ylabel('Pop-weighted')
        ax.set_title(title)
        ax.legend(fontsize=8)

    fig.suptitle('Cell-by-Cell: Pop-Weighted vs GBD-Direct (draw 0)', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Skipped (GBD artifact not available)')

### 2.8 Draw-Level Comparison (SBP Exposure)

Compare the uncertainty bands across all 1000 draws between the
two approaches.

In [ ]:
if gbd_exists:
    sbp_key = '/risk_factor/high_systolic_blood_pressure/exposure'
    pw_sbp = pd.read_hdf(POPWT_PATH, sbp_key)
    gbd_sbp = pd.read_hdf(GBD_PATH, sbp_key)
    draw_cols = [c for c in pw_sbp.columns if c.startswith('draw_')]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, sex in zip(axes, ['Female', 'Male']):
        for label, df, color in [
            ('Pop-weighted', pw_sbp.reset_index(), 'steelblue'),
            ('GBD-direct', gbd_sbp.reset_index(), 'coral'),
        ]:
            sub = df[df['sex'] == sex].sort_values('age_start')
            ages = sub['age_start'].values
            vals = sub[draw_cols].values
            median = np.median(vals, axis=1)
            p5 = np.percentile(vals, 5, axis=1)
            p95 = np.percentile(vals, 95, axis=1)
            ax.plot(ages, median, color=color, linewidth=2, label=f'{label} median')
            ax.fill_between(ages, p5, p95, alpha=0.2, color=color,
                            label=f'{label} 5-95%ile')
        ax.set_title(f'SBP Exposure - {sex}')
        ax.set_xlabel('Age')
        ax.set_ylabel('SBP (mmHg)')
        ax.legend(fontsize=8)
        ax.set_xlim(25, 100)

    fig.suptitle('SBP Exposure Uncertainty: Pop-Weighted vs GBD-Direct', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Skipped (GBD artifact not available)')

### 2.9 Medication Adherence Comparison

In [ ]:
if gbd_exists:
    for title, key in [
        ('SBP Medication Adherence', '/risk_factor/sbp_medication_adherence/exposure'),
        ('LDL-C Medication Adherence', '/risk_factor/ldlc_medication_adherence/exposure'),
    ]:
        pw_df = pd.read_hdf(POPWT_PATH, key)
        pw_df = pw_df[['draw_0']].rename(columns={'draw_0': 'value'}).reset_index()
        gbd_df = pd.read_hdf(GBD_PATH, key)
        gbd_df = gbd_df[['draw_0']].rename(columns={'draw_0': 'value'}).reset_index()

        if 'location' in pw_df.columns:
            pw_df = pw_df.drop(columns=['location'])
        if 'location' in gbd_df.columns:
            gbd_df = gbd_df.drop(columns=['location'])

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for ax, sex in zip(axes, ['Female', 'Male']):
            pw_sub = pw_df[(pw_df['sex'] == sex)].pivot(
                index='age_start', columns='parameter', values='value')
            gbd_sub = gbd_df[(gbd_df['sex'] == sex)].pivot(
                index='age_start', columns='parameter', values='value')

            # Plot difference
            diff = pw_sub - gbd_sub
            diff.plot.bar(ax=ax, colormap='Set2')
            ax.set_title(f'{title} Difference (PW - GBD) - {sex}')
            ax.set_xlabel('Age')
            ax.set_ylabel('Probability Difference')
            ax.legend(fontsize=7)
            ax.axhline(0, color='black', linewidth=0.5)
        plt.tight_layout()
        plt.show()
else:
    print('Skipped (GBD artifact not available)')

### 2.10 Full Key-by-Key Comparison Heatmap

For each key containing draws, compute the mean absolute relative
difference across all demographic cells and show as a heatmap.

In [ ]:
if gbd_exists:
    with pd.HDFStore(str(POPWT_PATH), 'r') as store:
        all_keys = sorted(store.keys())

    diffs = {}
    for key in all_keys:
        try:
            pw = pd.read_hdf(POPWT_PATH, key)
            gb = pd.read_hdf(GBD_PATH, key)
            if 'draw_0' not in pw.columns or 'draw_0' not in gb.columns:
                continue
            # Compare draw_0 only
            pw_vals = pw['draw_0'].values
            gb_vals = gb['draw_0'].values
            if len(pw_vals) != len(gb_vals):
                diffs[key] = np.nan
                continue
            mask = gb_vals != 0
            if mask.sum() == 0:
                continue
            rel = np.abs((pw_vals[mask] - gb_vals[mask]) / gb_vals[mask])
            diffs[key] = rel.mean() * 100
        except Exception:
            pass

    diff_series = pd.Series(diffs).sort_values(ascending=False)
    print('Mean Absolute Relative Difference (%) by Key (draw 0):')
    print('=' * 80)
    for k, v in diff_series.items():
        bar = '#' * int(min(v, 50))
        print(f'{k:70s} {v:7.2f}%  {bar}')
else:
    print('Skipped (GBD artifact not available)')

## Discussion

### Expected differences between the two approaches

1. **Population structure**: The GBD reports USA population directly,
   while the pop-weighted approach sums state populations. These should
   be very close but may differ slightly due to rounding or territory
   inclusion.

2. **Rates and exposures**: Population-weighted averaging of state rates
   is an approximation of the true national rate. It is exact when the
   relationship between population and the measure is linear. For
   non-linear quantities (e.g., prevalence derived from incidence and
   mortality), Jensen's inequality means the population-weighted average
   of state values may differ from the national value.

3. **PAFs**: These are computed via simulation, so differences arise from
   both the input data differences and Monte Carlo noise in the PAF
   calculation itself.

4. **Medication adherence/coverage**: The state-level medication data
   comes from local CSVs (`state_medication_real_data_v3.csv`). The
   pop-weighted approach averages these state values, while the GBD
   approach would need a USA entry added to the CSV (which would
   typically be the population-weighted average anyway).

5. **Location-independent keys**: Relative risks, TMREDs, mediation
   factors, and distributions are identical across all states and do
   not depend on location. These should be exactly identical between
   the two approaches.